# 01 - SVM (classical machine learning)
A Support Vector Machine with an RBF kernel, trained on the 784 raw pixels of each image.

## 1. Imports

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from torchvision import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.svm import SVC
import joblib

## 2. Settings
A fixed random seed so results can be repeated, and the folders where the model and results are saved.

In [2]:
SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR, MODELS_DIR, ART_DIR = ROOT / "data", ROOT / "app" / "models", ROOT / "artifacts"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(exist_ok=True)
np.random.seed(SEED)

## 3. Load the data
Download MNIST (first run only): 60,000 training images and 10,000 test images of handwritten digits.

In [3]:
train = datasets.MNIST(str(DATA_DIR), train=True, download=True)
test = datasets.MNIST(str(DATA_DIR), train=False, download=True)

## 4. Scale the pixels
Pixel values go from 0-255 to 0-1. The digit is white on a black background.

In [4]:
x_all, y_all = train.data.numpy().astype(np.float32) / 255, train.targets.numpy()
x_test, y_test = test.data.numpy().astype(np.float32) / 255, test.targets.numpy()

## 5. Split the data
The 60,000 training images become 54,000 for training and 6,000 for validation (used to compare settings). The 10,000 test images are only used once, at the end.

In [5]:
x_train, x_val, y_train, y_val = train_test_split(x_all, y_all, test_size=6000, stratify=y_all, random_state=SEED)
print(x_train.shape, x_val.shape, x_test.shape)

(54000, 28, 28) (6000, 28, 28) (10000, 28, 28)


## 6. Flatten the images
The SVM takes a flat row of 784 numbers per image instead of a 28x28 grid.

In [6]:
x_tr, x_v, x_te = (a.reshape(len(a), -1) for a in (x_train, x_val, x_test))
print(x_tr.shape)

(54000, 784)


## Helper: compute test metrics and save the results
Takes the test predictions and writes accuracy, precision, recall, F1, time, size and settings to `artifacts/`, so the comparison notebook can read them.

In [7]:
def save_results(name, model_path, pred_test, train_acc, val_acc, train_time, infer_ms, params, hyper, history=None):
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred_test, average="macro")
    res = dict(model=name, train_acc=train_acc, val_acc=val_acc, test_acc=accuracy_score(y_test, pred_test),
               macro_precision=p, macro_recall=r, macro_f1=f1, test_errors=int((pred_test != y_test).sum()),
               train_time_s=train_time, inference_ms_per_img=infer_ms, parameters=int(params),
               file_size_mb=model_path.stat().st_size / 1e6, hyperparameters=hyper)
    if history is not None:
        res["history"] = history
    (ART_DIR / f"{name.lower()}_results.json").write_text(json.dumps(res, indent=2, default=float))
    return res

## 7. Hyperparameter search
The SVM is slow, so each setting is trained on a random 10,000-image subsample. We try 3 values of C (how strictly it fits the training data) and 3 of gamma (how far each training image's influence reaches).

In [8]:
sub = np.random.RandomState(SEED).choice(len(x_tr), 10000, replace=False)
rows = []
for C in [1, 5, 10]:
    for gamma in [0.01, 0.02, 0.04]:
        m = SVC(kernel="rbf", C=C, gamma=gamma).fit(x_tr[sub], y_train[sub])
        rows.append(dict(C=C, gamma=gamma, val_acc=m.score(x_v, y_val)))

## 8. Pick the best setting
Sort by validation accuracy; the top row wins.

In [9]:
search = pd.DataFrame(rows).sort_values("val_acc", ascending=False).reset_index(drop=True)
best = search.iloc[0]
search

,C,gamma,val_acc
0,5,0.02,0.972333
1,10,0.02,0.972167
2,5,0.04,0.969167
3,10,0.04,0.969167
4,10,0.01,0.967833
5,5,0.01,0.967167
6,1,0.04,0.965667
7,1,0.02,0.965000
8,1,0.01,0.957833


## 9. Train the final model
Train on all 54,000 training images with the best C and gamma.

In [10]:
t0 = time.perf_counter()
svm = SVC(kernel="rbf", C=best["C"], gamma=best["gamma"]).fit(x_tr, y_train)
train_time = time.perf_counter() - t0
print("training time: %.1fs" % train_time)

training time: 200.9s


## 10. Predict on the test set
The first time the model sees the 10,000 test images.

In [11]:
pred_test = svm.predict(x_te)
print("test accuracy:", accuracy_score(y_test, pred_test))

test accuracy: 0.9851


## 11. Measure the prediction speed
Time for one image, averaged over 1,000 test images.

In [12]:
t0 = time.perf_counter()
svm.predict(x_te[:1000])
infer_ms = (time.perf_counter() - t0) * 1000 / 1000   # total ms for 1,000 images / 1,000
print("%.4f ms per image" % infer_ms)

4.2321 ms per image


## 12. Save the model
Compressed, because an SVM stores thousands of training images (the support vectors).

In [13]:
path = MODELS_DIR / "classical_svm.joblib"
joblib.dump(svm, path, compress=3)
print("saved", path, "-", round(path.stat().st_size / 1e6, 1), "MB")

saved /Users/seifebeid/Projects/minist/app/models/classical_svm.joblib - 5.8 MB


## 13. Accuracy on the training and validation data
Train accuracy is measured on 10,000 training images.

In [14]:
train_acc = svm.score(x_tr[:10000], y_train[:10000])
val_acc = svm.score(x_v, y_val)
print("train acc:", train_acc, "| val acc:", val_acc)

train acc: 0.9999 | val acc: 0.9841666666666666


## 14. Save the results
Writes `artifacts/svm_results.json` and shows the summary. For the SVM, "parameters" is the number of stored support-vector values.

In [15]:
res = save_results("SVM", path, pred_test, train_acc=train_acc, val_acc=val_acc, train_time=train_time, infer_ms=infer_ms,
                   params=svm.support_vectors_.size, hyper=dict(kernel="rbf", C=float(best["C"]), gamma=float(best["gamma"])))
pd.Series({k: v for k, v in res.items() if k != "hyperparameters"})

model                          SVM
train_acc                   0.9999
val_acc                   0.984167
test_acc                    0.9851
macro_precision            0.98508
macro_recall              0.984986
macro_f1                  0.985028
test_errors                    149
train_time_s            200.869222
inference_ms_per_img       4.23211
parameters                 9763152
file_size_mb              5.828041
dtype: object